# Emotion Recognition z tekstu — klasyczne ML, RNN, fine-tuning Transformerów i XAI

Notebook realizuje pełny projekt dla 3 datasetów: **GoEmotions**, **ISEAR** oraz **Sentiment and Emotion Analysis Dataset** z Kaggle.

Zakres:
- standaryzacja i połączenie datasetów,
- trening od zera: **SVM**, **Decision Tree**, **Naive Bayes**,
- trening od zera: **BiLSTM**, **BiGRU**,
- fine-tuning: **BERT**, **RoBERTa**, **DistilBERT**, **DistilRoBERTa**,
- uruchomienie gotowego modelu **Emotion RoBERTa**,
- metryki i wspólne wykresy porównawcze,
- XAI dobrane do typu modelu,
- porównanie podobieństwa/różnic wyjaśnień XAI,
- zapisywanie checkpointów do `./checkpoints`, wykresów do `./plots`, tabel wynikowych do `./reports`.

> Uwaga: notebook jest napisany jako praktyczny szablon badawczy. Wymaga dostosowania ścieżek/nazw kolumn w sekcji `CONFIG`, ponieważ wersje datasetów Kaggle często różnią się nazwami plików i kolumn.

In [ ]:

# Jeśli pracujesz w Colab/Kaggle, odkomentuj instalację zależności.
# !pip install -q pandas numpy scikit-learn matplotlib seaborn tqdm joblib kagglehub datasets transformers accelerate evaluate torch torchtext captum lime shap nltk wordcloud

In [1]:

import os
import re
import json
import math
import random
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_auc_score, balanced_accuracy_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr, kendalltau
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_context('notebook')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

/home/chemik/miniconda3/envs/ggsn_ex2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cuda'

In [2]:

# ===== CONFIG =====
PROJECT_ROOT = Path('.')
DATA_DIR = PROJECT_ROOT / 'data'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
PLOTS_DIR = PROJECT_ROOT / 'plots'
REPORTS_DIR = PROJECT_ROOT / 'reports'

for d in [DATA_DIR, CHECKPOINT_DIR, PLOTS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    # Ustaw lokalne ścieżki do pobranych datasetów Kaggle albo zostaw None i użyj sekcji kagglehub niżej.
    'goemotions_path': None,  # np. 'data/goemotions.csv'
    'isear_path': None,       # np. 'data/isear.csv'
    'sentiment_emotion_path': None,  # np. 'data/sentiment_emotion.csv'

    # Nazwy kolumn. Jeśli None, notebook spróbuje zgadnąć.
    'text_col_candidates': ['text', 'sentence', 'content', 'tweet', 'utterance', 'comment_text'],
    'label_col_candidates': ['emotion', 'label', 'sentiment', 'category', 'class'],

    # Czy łączyć wszystkie datasety w jeden wspólny benchmark.
    'combine_datasets': True,

    # Minimalna liczba przykładów na klasę po mapowaniu. Rzadkie klasy zostaną usunięte.
    'min_samples_per_class': 30,

    # Wspólny zestaw emocji. Możesz zawęzić albo rozszerzyć.
    'target_emotions': ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral'],

    # Parametry treningu.
    'test_size': 0.15,
    'val_size': 0.15,
    'max_features_tfidf': 50000,
    'max_seq_len_rnn': 80,
    'batch_size': 32,
    'rnn_epochs': 8,
    'transformer_epochs': 3,
    'learning_rate_rnn': 2e-3,
    'learning_rate_transformer': 2e-5,

    # Modele HF do fine-tuningu i model gotowy.
    'hf_models_to_finetune': {
        'BERT': 'bert-base-uncased',
        'RoBERTa': 'roberta-base',
        'DistilBERT': 'distilbert-base-uncased',
        'DistilRoBERTa': 'distilroberta-base',
    },
    'pretrained_emotion_roberta': 'SamLowe/roberta-base-go_emotions',

    # Ile przykładów użyć do XAI. Przy dużych modelach zwiększanie jest kosztowne.
    'xai_n_examples': 50,
    'xai_top_k': 10,
}

print(json.dumps(CONFIG, indent=2, default=str))

{
  "goemotions_path": null,
  "isear_path": null,
  "sentiment_emotion_path": null,
  "text_col_candidates": [
    "text",
    "sentence",
    "content",
    "tweet",
    "utterance",
    "comment_text"
  ],
  "label_col_candidates": [
    "emotion",
    "label",
    "sentiment",
    "category",
    "class"
  ],
  "combine_datasets": true,
  "min_samples_per_class": 30,
  "target_emotions": [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
    "neutral"
  ],
  "test_size": 0.15,
  "val_size": 0.15,
  "max_features_tfidf": 50000,
  "max_seq_len_rnn": 80,
  "batch_size": 32,
  "rnn_epochs": 8,
  "transformer_epochs": 3,
  "learning_rate_rnn": 0.002,
  "learning_rate_transformer": 2e-05,
  "hf_models_to_finetune": {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
    "DistilRoBERTa": "distilroberta-base"
  },
  "pretrained_emotion_roberta": "SamLowe/roberta-base-go_emotions",
  "xai_n_examples"

## 1. Pobieranie danych z Kaggle / kagglehub

Najbezpieczniej pobrać datasety ręcznie z Kaggle i wpisać ścieżki w `CONFIG`. Alternatywnie można użyć `kagglehub`. Slugi datasetów bywają różne — wpisz właściwe identyfikatory w komórce poniżej.

In [ ]:

# Opcjonalne pobieranie przez kagglehub.
# Po pobraniu sprawdź zawartość katalogu i ustaw CONFIG['..._path'] na konkretny plik CSV.

# import kagglehub
# KAGGLE_DATASETS = {
#     'goemotions': 'google-research-datasets/goemotions',
#     'isear': '.../...',
#     'sentiment_emotion': '.../...',
# }
# for name, slug in KAGGLE_DATASETS.items():
#     path = kagglehub.dataset_download(slug)
#     print(name, path)
#     print(list(Path(path).rglob('*'))[:20])

In [5]:

def find_csv_files(root: Path = Path('../data/project/clean')) -> List[Path]:
    return sorted(root.rglob('*.csv'))

print('CSV w data/:')
for p in find_csv_files():
    print(' -', p)

CSV w data/:
 - ../data/project/clean/.ipynb_checkpoints/combined_clean-checkpoint.csv
 - ../data/project/clean/.ipynb_checkpoints/goemotions_clean-checkpoint.csv
 - ../data/project/clean/.ipynb_checkpoints/isear_clean-checkpoint.csv
 - ../data/project/clean/combined_clean.csv
 - ../data/project/clean/goemotions_clean.csv
 - ../data/project/clean/isear_clean.csv


## 2. Wczytywanie i standaryzacja datasetów

Różne datasety mają różne etykiety. Funkcja `normalize_emotion` mapuje popularne warianty na wspólny zbiór emocji. W razie potrzeby dopisz mapowania dla swoich plików.

In [ ]:

EMOTION_MAP = {
    # podstawowe
    'anger': 'anger', 'angry': 'anger', 'rage': 'anger', 'annoyance': 'anger', 'annoyed': 'anger',
    'disgust': 'disgust', 'disgusted': 'disgust',
    'fear': 'fear', 'afraid': 'fear', 'scared': 'fear', 'anxiety': 'fear', 'nervousness': 'fear',
    'joy': 'joy', 'happy': 'joy', 'happiness': 'joy', 'amusement': 'joy', 'excitement': 'joy', 'love': 'joy', 'optimism': 'joy',
    'sadness': 'sadness', 'sad': 'sadness', 'grief': 'sadness', 'disappointment': 'sadness', 'remorse': 'sadness',
    'surprise': 'surprise', 'surprised': 'surprise', 'realization': 'surprise',
    'neutral': 'neutral', 'none': 'neutral', 'no emotion': 'neutral',

    # GoEmotions: przybliżone mapowania do 7 klas
    'admiration': 'joy', 'approval': 'joy', 'caring': 'joy', 'desire': 'joy', 'gratitude': 'joy', 'pride': 'joy', 'relief': 'joy',
    'curiosity': 'surprise', 'confusion': 'surprise',
    'disapproval': 'anger', 'embarrassment': 'sadness',
}

def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize_emotion(label: Any) -> Optional[str]:
    if pd.isna(label):
        return None
    s = str(label).strip().lower()
    s = re.sub(r'[_-]+', ' ', s)
    s = s.strip()
    return EMOTION_MAP.get(s, s if s in CONFIG['target_emotions'] else None)


def infer_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None


def load_generic_csv(path: Path, dataset_name: str, text_col: Optional[str] = None, label_col: Optional[str] = None) -> pd.DataFrame:
    df = pd.read_csv(path)
    text_col = text_col or infer_column(df, CONFIG['text_col_candidates'])
    label_col = label_col or infer_column(df, CONFIG['label_col_candidates'])
    if text_col is None or label_col is None:
        raise ValueError(f'Nie wykryto kolumn text/label dla {path}. Kolumny: {list(df.columns)}')
    out = pd.DataFrame({
        'text': df[text_col].map(clean_text),
        'emotion_raw': df[label_col],
        'dataset': dataset_name,
    })
    out['emotion'] = out['emotion_raw'].map(normalize_emotion)
    out = out.dropna(subset=['text', 'emotion'])
    out = out[out['text'].str.len() > 0].drop_duplicates(subset=['text', 'emotion'])
    return out


def load_goemotions(path: Path) -> pd.DataFrame:
    # Obsługa standardowych CSV GoEmotions z wieloma kolumnami 0/1 dla emocji lub prostego CSV text+label.
    df = pd.read_csv(path)
    text_col = infer_column(df, CONFIG['text_col_candidates']) or 'text'
    emotion_cols = [c for c in df.columns if c.lower() in EMOTION_MAP or c.lower() in CONFIG['target_emotions']]
    if len(emotion_cols) > 2 and text_col in df.columns:
        rows = []
        for _, r in df.iterrows():
            labels = []
            for c in emotion_cols:
                try:
                    if int(r[c]) == 1:
                        lab = normalize_emotion(c)
                        if lab:
                            labels.append(lab)
                except Exception:
                    pass
            # dla prostoty: jeśli wiele emocji mapuje się do różnych klas, bierzemy pierwszą według target_emotions
            labels = [x for x in labels if x in CONFIG['target_emotions']]
            if labels:
                chosen = sorted(labels, key=lambda x: CONFIG['target_emotions'].index(x))[0]
                rows.append({'text': clean_text(r[text_col]), 'emotion_raw': labels, 'emotion': chosen, 'dataset': 'goemotions'})
        return pd.DataFrame(rows).drop_duplicates(subset=['text', 'emotion'])
    return load_generic_csv(path, 'goemotions')


def build_dataset_from_paths() -> pd.DataFrame:
    parts = []
    if CONFIG['goemotions_path']:
        parts.append(load_goemotions(Path(CONFIG['goemotions_path'])))
    if CONFIG['isear_path']:
        parts.append(load_generic_csv(Path(CONFIG['isear_path']), 'isear'))
    if CONFIG['sentiment_emotion_path']:
        parts.append(load_generic_csv(Path(CONFIG['sentiment_emotion_path']), 'sentiment_emotion'))

    if not parts:
        raise FileNotFoundError('Ustaw ścieżki w CONFIG albo pobierz CSV do data/ i wczytaj je ręcznie.')
    data = pd.concat(parts, ignore_index=True)
    data = data[data['emotion'].isin(CONFIG['target_emotions'])]
    counts = data['emotion'].value_counts()
    keep = counts[counts >= CONFIG['min_samples_per_class']].index
    data = data[data['emotion'].isin(keep)].reset_index(drop=True)
    return data

# data = build_dataset_from_paths()
# display(data.head())
# print(data.shape)
# print(data['dataset'].value_counts())
# print(data['emotion'].value_counts())

In [ ]:

# Dla wygodnego uruchamiania: jeśli masz własny CSV już wczytany ręcznie,
# musi mieć kolumny: text, emotion, dataset.
# Przykład awaryjny, aby dalsze komórki nie odpalały się przypadkiem bez danych:

try:
    data
except NameError:
    data = None
    print('data=None. Ustaw CONFIG i odkomentuj build_dataset_from_paths().')

## 3. EDA i podział danych

In [ ]:

def plot_label_distribution(df: pd.DataFrame):
    plt.figure(figsize=(10, 5))
    ax = sns.countplot(data=df, x='emotion', order=df['emotion'].value_counts().index)
    ax.set_title('Rozkład klas emocji')
    ax.set_xlabel('Emocja')
    ax.set_ylabel('Liczba przykładów')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'label_distribution.png', dpi=200)
    plt.show()


def plot_dataset_distribution(df: pd.DataFrame):
    plt.figure(figsize=(10, 5))
    ax = sns.countplot(data=df, x='dataset', hue='emotion')
    ax.set_title('Rozkład emocji per dataset')
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Liczba przykładów')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'dataset_emotion_distribution.png', dpi=200)
    plt.show()

if data is not None:
    display(data.sample(min(5, len(data)), random_state=SEED))
    plot_label_distribution(data)
    plot_dataset_distribution(data)

In [ ]:

if data is not None:
    label_encoder = LabelEncoder()
    data['label'] = label_encoder.fit_transform(data['emotion'])
    label_names = list(label_encoder.classes_)
    num_labels = len(label_names)

    train_df, temp_df = train_test_split(
        data, test_size=CONFIG['test_size'] + CONFIG['val_size'], stratify=data['label'], random_state=SEED
    )
    rel_test = CONFIG['test_size'] / (CONFIG['test_size'] + CONFIG['val_size'])
    val_df, test_df = train_test_split(temp_df, test_size=rel_test, stratify=temp_df['label'], random_state=SEED)

    print(train_df.shape, val_df.shape, test_df.shape)
    print(label_names)
    joblib.dump(label_encoder, CHECKPOINT_DIR / 'label_encoder.joblib')

## 4. Funkcje ewaluacji i wizualizacji

In [ ]:

all_results = []
all_predictions = {}


def evaluate_predictions(model_name: str, y_true: np.ndarray, y_pred: np.ndarray, y_proba: Optional[np.ndarray] = None) -> Dict[str, Any]:
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    w_precision, w_recall, w_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    result = {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_precision': precision,
        'macro_recall': recall,
        'macro_f1': f1,
        'weighted_precision': w_precision,
        'weighted_recall': w_recall,
        'weighted_f1': w_f1,
    }
    if y_proba is not None and len(np.unique(y_true)) > 2:
        try:
            result['roc_auc_ovr_macro'] = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
        except Exception:
            result['roc_auc_ovr_macro'] = np.nan
    return result


def save_classification_report(model_name, y_true, y_pred):
    report = classification_report(y_true, y_pred, target_names=label_names, zero_division=0, output_dict=True)
    pd.DataFrame(report).T.to_csv(REPORTS_DIR / f'{model_name}_classification_report.csv')
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))


def plot_confusion(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
    plt.title(f'Confusion matrix — {model_name}')
    plt.xlabel('Predykcja')
    plt.ylabel('Prawda')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f'confusion_{model_name}.png', dpi=200)
    plt.show()


def add_result(model_name, y_true, y_pred, y_proba=None):
    res = evaluate_predictions(model_name, y_true, y_pred, y_proba)
    all_results.append(res)
    all_predictions[model_name] = {'y_true': np.array(y_true), 'y_pred': np.array(y_pred), 'y_proba': y_proba}
    save_classification_report(model_name, y_true, y_pred)
    plot_confusion(model_name, y_true, y_pred)
    return res

## 5. Modele klasyczne: SVM, Decision Tree, Naive Bayes

In [ ]:

classic_models = {}

if data is not None:
    X_train, y_train = train_df['text'].values, train_df['label'].values
    X_test, y_test = test_df['text'].values, test_df['label'].values

    model_specs = {
        'SVM': CalibratedClassifierCV(LinearSVC(class_weight='balanced', random_state=SEED), cv=3),
        'DecisionTree': DecisionTreeClassifier(max_depth=None, class_weight='balanced', random_state=SEED),
        'NaiveBayes': MultinomialNB(),
    }

    for name, clf in model_specs.items():
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(max_features=CONFIG['max_features_tfidf'], ngram_range=(1,2), min_df=2)),
            ('clf', clf)
        ])
        print(f'Training {name}...')
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        proba = pipe.predict_proba(X_test) if hasattr(pipe, 'predict_proba') else None
        classic_models[name] = pipe
        joblib.dump(pipe, CHECKPOINT_DIR / f'{name}.joblib')
        add_result(name, y_test, pred, proba)

## 6. Modele rekurencyjne od zera: BiLSTM i BiGRU

In [ ]:

class TextVocab:
    def __init__(self, max_size=50000, min_freq=2):
        self.max_size = max_size
        self.min_freq = min_freq
        self.stoi = {'<pad>': 0, '<unk>': 1}
        self.itos = ['<pad>', '<unk>']

    def tokenize(self, text):
        return re.findall(r"\w+|[^\w\s]", str(text).lower())

    def fit(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(self.tokenize(t))
        for tok, freq in counter.most_common(self.max_size - 2):
            if freq >= self.min_freq:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)
        return self

    def encode(self, text, max_len):
        ids = [self.stoi.get(tok, 1) for tok in self.tokenize(text)[:max_len]]
        ids += [0] * (max_len - len(ids))
        return ids

class RNNDataset(Dataset):
    def __init__(self, df, vocab, max_len):
        self.texts = df['text'].tolist()
        self.labels = df['label'].values.astype(np.int64)
        self.vocab = vocab
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.vocab.encode(self.texts[idx], self.max_len), dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class BiRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, rnn_type='lstm', dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if rnn_type == 'lstm':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        elif rnn_type == 'gru':
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        else:
            raise ValueError(rnn_type)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)
    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        out, hidden = self.rnn(emb)
        mask = (input_ids != 0).unsqueeze(-1)
        out = out.masked_fill(~mask, 0.0)
        pooled = out.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc(self.dropout(pooled))


def train_rnn_model(model_name, rnn_type):
    vocab = TextVocab(max_size=50000, min_freq=2).fit(train_df['text'])
    train_ds = RNNDataset(train_df, vocab, CONFIG['max_seq_len_rnn'])
    val_ds = RNNDataset(val_df, vocab, CONFIG['max_seq_len_rnn'])
    test_ds = RNNDataset(test_df, vocab, CONFIG['max_seq_len_rnn'])
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'])
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'])

    model = BiRNNClassifier(len(vocab.itos), 128, 128, num_labels, rnn_type=rnn_type).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate_rnn'])
    criterion = nn.CrossEntropyLoss()
    best_val_f1 = -1

    for epoch in range(CONFIG['rnn_epochs']):
        model.train()
        losses = []
        for batch in tqdm(train_loader, desc=f'{model_name} epoch {epoch+1}'):
            ids = batch['input_ids'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            optim.zero_grad()
            logits = model(ids)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            losses.append(loss.item())

        y_val, p_val = predict_rnn(model, val_loader)
        f1 = precision_recall_fscore_support(y_val, p_val, average='macro', zero_division=0)[2]
        print(f'{model_name} epoch={epoch+1} loss={np.mean(losses):.4f} val_macro_f1={f1:.4f}')
        if f1 > best_val_f1:
            best_val_f1 = f1
            torch.save({'model_state': model.state_dict(), 'vocab': vocab, 'config': CONFIG}, CHECKPOINT_DIR / f'{model_name}.pt')

    ckpt = torch.load(CHECKPOINT_DIR / f'{model_name}.pt', map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    y_true, y_pred, y_proba = predict_rnn(model, test_loader, return_proba=True)
    add_result(model_name, y_true, y_pred, y_proba)
    return model, vocab

@torch.no_grad()
def predict_rnn(model, loader, return_proba=False):
    model.eval()
    y_true, y_pred, y_proba = [], [], []
    for batch in loader:
        ids = batch['input_ids'].to(DEVICE)
        labels = batch['label'].cpu().numpy()
        logits = model(ids)
        proba = torch.softmax(logits, dim=-1).cpu().numpy()
        pred = proba.argmax(axis=1)
        y_true.extend(labels)
        y_pred.extend(pred)
        y_proba.extend(proba)
    if return_proba:
        return np.array(y_true), np.array(y_pred), np.array(y_proba)
    return np.array(y_true), np.array(y_pred)

rnn_models = {}
if data is not None:
    rnn_models['BiLSTM'] = train_rnn_model('BiLSTM', 'lstm')
    rnn_models['BiGRU'] = train_rnn_model('BiGRU', 'gru')

## 7. Fine-tuning Transformerów: BERT, RoBERTa, DistilBERT, DistilRoBERTa

In [ ]:

try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, pipeline
    import evaluate
    HF_AVAILABLE = True
except Exception as e:
    HF_AVAILABLE = False
    print('Transformers/evaluate niedostępne:', e)

class HFDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        item = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length)
        item['labels'] = self.labels[idx]
        return item


def compute_metrics_hf(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': accuracy_score(labels, preds), 'macro_precision': p, 'macro_recall': r, 'macro_f1': f1}


def finetune_hf_model(model_alias, model_name):
    out_dir = CHECKPOINT_DIR / f'hf_{model_alias}'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label={i: lab for i, lab in enumerate(label_names)},
        label2id={lab: i for i, lab in enumerate(label_names)},
    )
    train_ds = HFDataset(train_df, tokenizer)
    val_ds = HFDataset(val_df, tokenizer)
    test_ds = HFDataset(test_df, tokenizer)
    args = TrainingArguments(
        output_dir=str(out_dir),
        evaluation_strategy='epoch',
        save_strategy='epoch',
        learning_rate=CONFIG['learning_rate_transformer'],
        per_device_train_batch_size=CONFIG['batch_size'],
        per_device_eval_batch_size=CONFIG['batch_size'],
        num_train_epochs=CONFIG['transformer_epochs'],
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        logging_steps=50,
        report_to='none',
        seed=SEED,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics_hf,
    )
    trainer.train()
    trainer.save_model(str(out_dir / 'best'))
    tokenizer.save_pretrained(str(out_dir / 'best'))
    pred = trainer.predict(test_ds)
    y_true = pred.label_ids
    y_proba = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    y_pred = y_proba.argmax(axis=1)
    add_result(model_alias, y_true, y_pred, y_proba)
    return trainer, tokenizer

hf_models = {}
if data is not None and HF_AVAILABLE:
    for alias, model_name in CONFIG['hf_models_to_finetune'].items():
        print('Fine-tuning:', alias, model_name)
        hf_models[alias] = finetune_hf_model(alias, model_name)

## 8. Gotowy model Emotion RoBERTa z internetu

Ten model może mieć inny zestaw etykiet niż nasz benchmark. Komórka mapuje predykcje modelu do wspólnych emocji przez `normalize_emotion`. Predykcje, których nie da się zmapować, są ustawiane jako `neutral` albo najbliższa dostępna klasa.

In [ ]:

def run_pretrained_emotion_roberta():
    clf = pipeline('text-classification', model=CONFIG['pretrained_emotion_roberta'], top_k=None, device=0 if DEVICE=='cuda' else -1)
    preds, probas = [], []
    for text in tqdm(test_df['text'].tolist(), desc='Pretrained Emotion RoBERTa'):
        outputs = clf(text)[0]
        scores = np.zeros(num_labels, dtype=float)
        for o in outputs:
            lab = normalize_emotion(o['label'])
            if lab in label_names:
                scores[label_names.index(lab)] += float(o['score'])
        if scores.sum() == 0:
            # fallback: neutral, jeśli istnieje, inaczej klasa 0
            fallback = label_names.index('neutral') if 'neutral' in label_names else 0
            scores[fallback] = 1.0
        scores = scores / scores.sum()
        probas.append(scores)
        preds.append(scores.argmax())
    y_true = test_df['label'].values
    y_pred = np.array(preds)
    y_proba = np.array(probas)
    add_result('PretrainedEmotionRoBERTa', y_true, y_pred, y_proba)

if data is not None and HF_AVAILABLE:
    run_pretrained_emotion_roberta()

## 9. Wspólne porównanie wyników modeli

In [ ]:

def make_results_table():
    results_df = pd.DataFrame(all_results).sort_values('macro_f1', ascending=False)
    results_df.to_csv(REPORTS_DIR / 'all_model_results.csv', index=False)
    return results_df

if all_results:
    results_df = make_results_table()
    display(results_df)

    metrics = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']
    plot_df = results_df.melt(id_vars='model', value_vars=metrics, var_name='metric', value_name='score')
    plt.figure(figsize=(14, 6))
    sns.barplot(data=plot_df, x='model', y='score', hue='metric')
    plt.title('Porównanie metryk modeli')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'model_metrics_comparison.png', dpi=200)
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.heatmap(results_df.set_index('model')[metrics], annot=True, fmt='.3f', cmap='viridis')
    plt.title('Heatmapa metryk')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'model_metrics_heatmap.png', dpi=200)
    plt.show()

## 10. XAI — wyjaśnienia dla różnych rodzin modeli

Zastosowane podejścia:
- modele klasyczne TF-IDF: współczynniki / ważność cech, permutation importance, LIME,
- RNN: Integrated Gradients przez Captum na embeddingach,
- Transformery fine-tunowane/gotowe: LIME + opcjonalnie Integrated Gradients/Captum,
- wspólna reprezentacja wyjaśnień: słownik `token -> attribution`, co pozwala porównać podobieństwo XAI.

In [ ]:

xai_explanations = defaultdict(dict)  # model_name -> sample_index -> {token: score}

sample_xai_df = test_df.sample(min(CONFIG['xai_n_examples'], len(test_df)), random_state=SEED).reset_index(drop=True) if data is not None else None


def normalize_attr_dict(d: Dict[str, float]) -> Dict[str, float]:
    total = sum(abs(v) for v in d.values())
    if total == 0:
        return d
    return {k: float(v) / total for k, v in d.items()}


def top_tokens_plot(model_name, attr_dict, filename_suffix, top_k=20):
    items = sorted(attr_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:top_k]
    if not items:
        return
    toks, vals = zip(*items)
    plt.figure(figsize=(10, max(4, 0.35*len(toks))))
    sns.barplot(x=list(vals), y=list(toks), orient='h')
    plt.title(f'Najważniejsze tokeny XAI — {model_name}')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f'xai_top_tokens_{model_name}_{filename_suffix}.png', dpi=200)
    plt.show()


def classic_global_feature_importance(model_name, pipe, top_k=30):
    vectorizer = pipe.named_steps['tfidf']
    clf = pipe.named_steps['clf']
    features = np.array(vectorizer.get_feature_names_out())
    global_scores = defaultdict(float)
    # Linear SVM CalibratedClassifierCV ma listę estimatorów bazowych
    if hasattr(clf, 'calibrated_classifiers_'):
        coefs = []
        for cc in clf.calibrated_classifiers_:
            base = getattr(cc, 'estimator', None) or getattr(cc, 'base_estimator', None)
            if hasattr(base, 'coef_'):
                coefs.append(base.coef_)
        if coefs:
            coef = np.mean(np.stack(coefs), axis=0)
            scores = np.mean(np.abs(coef), axis=0)
            global_scores = dict(zip(features, scores))
    elif hasattr(clf, 'feature_importances_'):
        global_scores = dict(zip(features, clf.feature_importances_))
    elif hasattr(clf, 'feature_log_prob_'):
        scores = np.std(clf.feature_log_prob_, axis=0)
        global_scores = dict(zip(features, scores))
    global_scores = normalize_attr_dict(global_scores)
    top_tokens_plot(model_name, global_scores, 'global', top_k=top_k)
    with open(REPORTS_DIR / f'xai_global_{model_name}.json', 'w') as f:
        json.dump(global_scores, f, indent=2)
    return global_scores

if data is not None:
    for name, pipe in classic_models.items():
        g = classic_global_feature_importance(name, pipe)
        for i, row in sample_xai_df.iterrows():
            # lokalna aproksymacja: TF-IDF tekstu * globalna ważność cechy
            vec = pipe.named_steps['tfidf'].transform([row['text']])
            feats = pipe.named_steps['tfidf'].get_feature_names_out()
            local = {feats[j]: float(vec[0, j]) * g.get(feats[j], 0.0) for j in vec.nonzero()[1]}
            xai_explanations[name][i] = normalize_attr_dict(local)

In [ ]:

# LIME dla modeli mających predict_proba. Działa również jako model-agnostic XAI dla klasycznych modeli.
try:
    from lime.lime_text import LimeTextExplainer
    LIME_AVAILABLE = True
except Exception as e:
    LIME_AVAILABLE = False
    print('LIME niedostępne:', e)

if data is not None and LIME_AVAILABLE:
    explainer = LimeTextExplainer(class_names=label_names)
    for name, pipe in classic_models.items():
        if not hasattr(pipe, 'predict_proba'):
            continue
        for i, row in tqdm(sample_xai_df.iterrows(), total=len(sample_xai_df), desc=f'LIME {name}'):
            exp = explainer.explain_instance(row['text'], pipe.predict_proba, num_features=CONFIG['xai_top_k'], top_labels=1)
            pred_label = int(pipe.predict([row['text']])[0])
            attr = dict(exp.as_list(label=pred_label))
            xai_explanations[f'LIME_{name}'][i] = normalize_attr_dict(attr)

In [ ]:

# Integrated Gradients dla RNN przez Captum.
try:
    from captum.attr import LayerIntegratedGradients
    CAPTUM_AVAILABLE = True
except Exception as e:
    CAPTUM_AVAILABLE = False
    print('Captum niedostępny:', e)


def explain_rnn_integrated_gradients(model, vocab, text, target_label):
    model.eval()
    ids = torch.tensor([vocab.encode(text, CONFIG['max_seq_len_rnn'])], dtype=torch.long).to(DEVICE)
    tokens = [vocab.itos[i] if i < len(vocab.itos) else '<unk>' for i in ids[0].detach().cpu().numpy()]
    lig = LayerIntegratedGradients(lambda x: model(x), model.embedding)
    baseline = torch.zeros_like(ids).to(DEVICE)
    attributions, delta = lig.attribute(ids, baselines=baseline, target=int(target_label), return_convergence_delta=True)
    scores = attributions.sum(dim=-1).squeeze(0).detach().cpu().numpy()
    attr = defaultdict(float)
    for tok, score in zip(tokens, scores):
        if tok not in ['<pad>', '<unk>']:
            attr[tok] += float(score)
    return normalize_attr_dict(dict(attr))

if data is not None and CAPTUM_AVAILABLE:
    for name, (model, vocab) in rnn_models.items():
        for i, row in tqdm(sample_xai_df.iterrows(), total=len(sample_xai_df), desc=f'IG {name}'):
            xai_explanations[f'IG_{name}'][i] = explain_rnn_integrated_gradients(model, vocab, row['text'], row['label'])
        # pokaż pierwszy przykład
        if len(sample_xai_df):
            top_tokens_plot(f'IG_{name}', xai_explanations[f'IG_{name}'][0], 'sample0')

In [ ]:

# LIME dla Transformerów: kosztowne, więc domyślnie tylko xai_n_examples.
def make_hf_predict_proba(trainer, tokenizer):
    def predict_proba(texts):
        ds = HFDataset(pd.DataFrame({'text': texts, 'label': [0]*len(texts)}), tokenizer)
        pred = trainer.predict(ds)
        return torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    return predict_proba

if data is not None and HF_AVAILABLE and LIME_AVAILABLE:
    explainer = LimeTextExplainer(class_names=label_names)
    for name, (trainer, tokenizer) in hf_models.items():
        predict_proba = make_hf_predict_proba(trainer, tokenizer)
        for i, row in tqdm(sample_xai_df.iterrows(), total=len(sample_xai_df), desc=f'LIME {name}'):
            exp = explainer.explain_instance(row['text'], predict_proba, num_features=CONFIG['xai_top_k'], top_labels=1)
            pred_label = int(np.argmax(predict_proba([row['text']])[0]))
            xai_explanations[f'LIME_{name}'][i] = normalize_attr_dict(dict(exp.as_list(label=pred_label)))

## 11. Porównanie podobieństwa wyjaśnień XAI

Metryki:
- **Jaccard@K** — nakładanie się top-K najważniejszych tokenów,
- **Spearman** — korelacja rang ważności tokenów,
- **Cosine similarity** — podobieństwo wektorów atrybucji,
- **Sign agreement** — zgodność znaków wpływu tokenów.

In [ ]:

def vectorize_two_attr(a: Dict[str, float], b: Dict[str, float]):
    keys = sorted(set(a) | set(b))
    va = np.array([a.get(k, 0.0) for k in keys])
    vb = np.array([b.get(k, 0.0) for k in keys])
    return keys, va, vb


def jaccard_top_k(a, b, k=10):
    ta = set([x for x, _ in sorted(a.items(), key=lambda z: abs(z[1]), reverse=True)[:k]])
    tb = set([x for x, _ in sorted(b.items(), key=lambda z: abs(z[1]), reverse=True)[:k]])
    if not ta and not tb:
        return np.nan
    return len(ta & tb) / len(ta | tb)


def attr_similarity(a, b, k=10):
    keys, va, vb = vectorize_two_attr(a, b)
    if len(keys) == 0:
        return {'jaccard_top_k': np.nan, 'spearman': np.nan, 'kendall': np.nan, 'cosine': np.nan, 'sign_agreement': np.nan}
    cos = cosine_similarity([va], [vb])[0, 0] if np.linalg.norm(va) > 0 and np.linalg.norm(vb) > 0 else np.nan
    sp = spearmanr(va, vb).correlation if len(keys) > 1 else np.nan
    kt = kendalltau(va, vb).correlation if len(keys) > 1 else np.nan
    nz = (va != 0) | (vb != 0)
    sign_agree = np.mean(np.sign(va[nz]) == np.sign(vb[nz])) if nz.any() else np.nan
    return {
        'jaccard_top_k': jaccard_top_k(a, b, k),
        'spearman': sp,
        'kendall': kt,
        'cosine': cos,
        'sign_agreement': sign_agree,
    }


def compare_xai_methods():
    rows = []
    methods = sorted(xai_explanations.keys())
    for m1_i, m1 in enumerate(methods):
        for m2 in methods[m1_i+1:]:
            common_ids = sorted(set(xai_explanations[m1]) & set(xai_explanations[m2]))
            if not common_ids:
                continue
            sims = [attr_similarity(xai_explanations[m1][i], xai_explanations[m2][i], CONFIG['xai_top_k']) for i in common_ids]
            row = {'method_a': m1, 'method_b': m2, 'n': len(common_ids)}
            for metric in sims[0].keys():
                row[metric] = np.nanmean([s[metric] for s in sims])
            rows.append(row)
    return pd.DataFrame(rows)

if xai_explanations:
    xai_sim_df = compare_xai_methods()
    xai_sim_df.to_csv(REPORTS_DIR / 'xai_similarity.csv', index=False)
    display(xai_sim_df.sort_values('jaccard_top_k', ascending=False).head(30))

    for metric in ['jaccard_top_k', 'spearman', 'cosine', 'sign_agreement']:
        pivot = pd.DataFrame(index=sorted(xai_explanations.keys()), columns=sorted(xai_explanations.keys()), dtype=float)
        np.fill_diagonal(pivot.values, 1.0)
        for _, r in xai_sim_df.iterrows():
            pivot.loc[r['method_a'], r['method_b']] = r[metric]
            pivot.loc[r['method_b'], r['method_a']] = r[metric]
        plt.figure(figsize=(12, 10))
        sns.heatmap(pivot.astype(float), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
        plt.title(f'Podobieństwo XAI — {metric}')
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f'xai_similarity_{metric}.png', dpi=200)
        plt.show()

## 12. Eksport artefaktów i podsumowanie

In [ ]:

summary = {
    'n_samples': int(len(data)) if data is not None else None,
    'labels': label_names if data is not None else None,
    'models_evaluated': list(all_predictions.keys()),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'plots_dir': str(PLOTS_DIR),
    'reports_dir': str(REPORTS_DIR),
}
with open(REPORTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print('
Gotowe artefakty:')
print('Checkpoints:', list(CHECKPOINT_DIR.glob('*'))[:20])
print('Plots:', list(PLOTS_DIR.glob('*'))[:20])
print('Reports:', list(REPORTS_DIR.glob('*'))[:20])

## 13. Proponowane rozszerzenia do pracy/projektu

- uruchomić osobne benchmarki per dataset oraz benchmark połączony,
- dodać wariant multi-label dla GoEmotions zamiast sprowadzania do single-label,
- dodać macro/micro F1 dla każdego datasetu oddzielnie,
- sprawdzić stabilność XAI między seedami,
- porównać XAI z ręcznymi słownikami emocji, np. NRC Emotion Lexicon,
- zrobić ablation: bez neutral, bez rzadkich klas, bez mapowania GoEmotions do 7 klas.